# Data Preparation for E2SFCA Analysis
## Emergency Obstetric Care Accessibility - Kano State, Nigeria

> Note: This notebook requires the [environment dependencies](requirements.txt) to be installed
> as well as either an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local instance of the ORS server.

## Model Summary:

This notebook provides the means to generate a dataset that is described in the [model documentation](../kano/dataset-interpretability.md).

## Workflow Summary:

The notebook gives an overview of the distribution of centres offering EmOC in the city, their classification and how they can be accessed during an emergency. Open source data from OpenStreetMap and tools (such as the openrouteservice) were used to create accessibility measures. Spatial analysis and other data analytics functions led to generating outputs within the 100x100m grid cells that categorised them into three levels: low, medium, and high.

* **Preprocessing**: Get data for EmOC facilities.
* **Analysis for Offer**:
    * Filter or classify EmOC facilities based on discussed criteria.
    * Visualise EmOC faccilities in their categories.
* **Analysis for Accessibility**:
    * Compute travel times to facilities using openrouteservice API or other routing services.
    * Generate areas for low, medium and high categories based on discussed criteria.
* **Analysis for Demmand**:
    * Downscale the popluation data to the 100x100m grid cells.
    * Derive socio-economic descriptors based on discussed criteria.

* **Result**: Generate results as GIS-compatible files.

#  Workflow

Make sure you have the required packages installed. You can install them using pip:

```bash
pip install -r requirements.txt
```

This study integrates various Python geospatial analysis libraries and packages to support spatial data processing, visualization, and isochrone generation. The os module is used to interact with the operating system, managing file paths and reading environment variables such as API keys. folium library along with its MarkerCluster plugin, facilitates the creation of interactive maps for visualizing large-scale geospatial data. The openrouteservice.client serves as an interface to the OpenRouteService API, enabling the extraction of isochrones. pandas library for data analysis, provides functions for analyzing, cleaning, exploring, and manipulating data, while fiona supports reading and writing real-world data using multi-layered GIS formats, such as shapefiles. The shapely package is employed for the manipulation and analysis of planar geometric objects.

## Setting up the virtual environment

```bash
# Create a new virtual environment
python -m venv .venv
activate .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [3]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd

import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point
from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

### Setting up relevant processing folders

There are different data sources used across the notebook. To handle these data sets, it is recommended to use three directories for input, temp and output data. Some of the files are related to healthcare facilities, population data. The healthcare facilities data is usualy the result of gathering global or national datasets and then carrying out local validation according to the local context. 

Despite being official, administrative boundaries may not reflect the actual patterns of human settlement or economic activity. Therefore, the team used the Functional Urban Area (FUA) as a complementary definition of the study areas. The FUA is defined by [the Joint Research Centre of the European Commission](https://commission.europa.eu/about/departments-and-executive-agencies/joint-research-centre_en) as the actual urban sprawl and human activities, encompassing the core city and economically or socially integrated surrounding regions. The FUA was obtained from [the Global Human Settlement Layer (GHSL) ](https://human-settlement.emergency.copernicus.eu/)dataset, which provides spatial data for functional urban areas worldwide. 

The following datasets are considered as input data for the analysis:


* [Datasets of health facilities](../scripts/Kano/data-inputs/healthcare_facilities.geojson)
* [Population: Women in childbearing age](../scripts/Kano/data-inputs/kano_nga_f_15_49_2015_1km.tif) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447)
* [Study Area](../../../docs/study-areas/grid-boundary-kano.gpkg) defined by the IDEAMAPS team

In [4]:
# Set paths to access Kano data
# Define directories
data_inputs = '../Kano/Data/raw/'
data_temp = '../Kano/Data/processed/'
model_outputs = '../Kano/Data/outputs/'

## 1. Data Collection

### Validated healthcare facilities - (Supply/Offer)
For Kano, the classification for validation was determined with the assistance of local experts, based on data obtained from the [datasets of health facilities](https://doi.org/10.6084/m9.figshare.22689667.v2).

In [5]:
# Load the healthcare facilities data
healthcare_facilities_validated = gpd.read_file(data_inputs + 'healthcare_facilities.geojson')

# Display basic info
print(f"\n✓ Loaded {len(healthcare_facilities_validated)} healthcare facilities")
print(f"CRS: {healthcare_facilities_validated.crs}")

healthcare_facilities_validated.head()


✓ Loaded 145 healthcare facilities
CRS: EPSG:4326


,orig_order,state,lga,ward,urban_conurb,uid,facility_code,ontime_code,facility_name,reg_number,...,longitude,operation_status,registration_status,license_status,created,last_updated,last_updated_ontime,Local_Validation,hcf_id,geometry
0,1210,9,Fagge,Kwachiri,9,12757068.0,19/12/1/2/1/0004,100904010,465 Nigerian Airforce Base Hospital,NaN,...,8.531780,Operational,Registered,Licensed,2018-01-01 01:01:01,2019-12-30 22:56:13,28/09/2022 09:00,Public Comprehensive EmOC,25,POINT (8.53178 12.04531)
1,1208,9,Fagge,Fagge D 2,9,23158449.0,19/12/1/1/1/0001,100904008,Abubakar Imam Urology Centre,NaN,...,8.525738,Operational,Registered,Licensed,2018-01-01 01:01:01,2019-12-30 22:58:50,28/09/2022 09:00,No EmOC,23,POINT (8.52574 12.01474)
2,1302,9,Tarauni,Gyadi-Gyadi Arewa,9,40297833.0,19/21/1/1/2/0004,100911002,Access Clinic,BN 0013966,...,8.541140,Operational,Registered,Licensed,2018-01-01 01:01:01,2020-01-10 20:54:32,28/09/2022 09:00,Private Comprehensive EmOC,114,POINT (8.54114 11.97802)
3,1277,9,Nasarawa,Tudun Wada (NSR),9,42838223.0,19/31/1/2/2/0001,100910010,Ahmadiyya Muslim Hospital,NaN,...,8.548216,Operational,Registered,Licensed,2018-01-01 01:01:01,2020-01-06 14:14:49,28/09/2022 09:00,Private Comprehensive EmOC,92,POINT (8.54822 12.00668)
4,1327,9,Tarauni,Babban Giji,9,NaN,NaN,100911027,Ahmed Memorial Clinic and Maternity,NaN,...,8.535700,Operational,NaN,NaN,NaT,NaT,28/09/2022 09:00,Private Comprehensive EmOC,139,POINT (8.5357 11.96691)


### Population Grid Data (Demand)
This data originally comes as a grid (1km resolution) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447) to transform it into a 100x100m grid, we use a procedure explained below. 

Note: explain the process to scale down the population data. 
note: explain the rational for female population between 15-49 years old.

In [ ]:
# Load the study area grid
study_area = gpd.read_file(data_inputs + '100mGrid.gpkg')

# Load the population raster data for Nigeria
raster_path = data_inputs + 'nga_f_15_49_2015_1km.tif'

Clipping the population data to our study area

In [ ]:
# Clip the population raster to the study area of Kano State
with rasterio.open(raster_path) as dataset:
    geometries = [study_area.geometry.unary_union.__geo_interface__]
    clipped_image, clipped_transform = mask(dataset, geometries, crop=True)
    band1 = clipped_image[0] # Read the first band of the raster

# Update the metadata for the clipped raster
out_meta = dataset.meta.copy()
out_meta.update({
        "height": clipped_image.shape[1],
        "width": clipped_image.shape[2],
        "transform": clipped_transform
    })

# Save the clipped raster to a new file
with rasterio.open(data_inputs + 'kano_nga_f_15_49_2015_1km.tif', "w", **out_meta) as dest:
    dest.write(clipped_image)

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_96338/2284915905.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometries = [study_area.geometry.unary_union.__geo_interface__]


Calculating the centroids for grid cells

In [ ]:
# Extract the population values and their corresponding coordinates from the clipped raster
rows, cols = np.where(band1 > 0)
grid_cells = [clipped_transform * (col + 0.5, row + 0.5) for row, col in zip(rows, cols)]
population_values = band1[rows, cols]

# Create a GeoDataFrame for the population centroids
grid_df = pd.DataFrame(grid_cells, columns=["longitude", "latitude"])
grid_df["population"] = population_values
grid_df["rowid"] = range(1, len(grid_df) + 1)
population_centroids_gdf = gpd.GeoDataFrame(grid_df, geometry=[Point(xy) for xy in zip(grid_df["longitude"], grid_df["latitude"])])
population_centroids_gdf.set_crs("EPSG:4326", inplace=True)

# Save the population centroids GeoDataFrame to a file
population_centroids_gdf.to_file(data_temp + "population_centroids.gpkg", driver="GPKG")
population_centroids_gdf

### Adding population data at 1km grid to 100m grid

In [12]:
# reading in geotiff file as numpy array
def read_tif(file: Path):
    if not file.exists():
        raise FileNotFoundError(f'File {file} not found')

    with rasterio.open(file) as dataset:
        arr = dataset.read()  # (bands X height X width)
        nodata = dataset.nodata
        transform = dataset.transform
        crs = dataset.crs

    # Replace NoData value with NaN
    if nodata is not None:
        arr[arr == nodata] = np.nan

    return arr.transpose((1, 2, 0)), transform, crs

def raster2vector(arr, transform, crs) -> gpd.GeoDataFrame:
    height, width, bands = arr.shape

    # Generate pixel coordinates
    geometries = []
    pixel_values = []

    for row in range(height):
        for col in range(width):
            x_min, y_max = transform * (col, row)  # Top-left corner
            x_max, y_min = transform * (col + 1, row + 1)  # Bottom-right corner

            pixel_value = arr[row, col].tolist()[0]  # Convert numpy array to list
            polygon = Polygon([(x_min, y_max), (x_max, y_max), (x_max, y_min), (x_min, y_min)])

            geometries.append(polygon)
            pixel_values.append(pixel_value)

    # Convert to DataFrame
    gdf = gpd.GeoDataFrame({'pop_grid_pop': pixel_values, 'geometry': geometries}, crs=crs)

    return gdf

epsg = 'EPSG:32632'

In [ ]:
# Preparing grid
grid_file = data_inputs + 'grid-boundary-kano.gpkg'
grid = gpd.read_file(grid_file)
grid = grid.to_crs(epsg)
grid['grid_id'] = range(len(grid))
grid = grid[['grid_id', 'geometry','rowid', 'latitude', 'lat_min', 'lat_max', 'longitude', 'lon_min','lon_max']].set_geometry('geometry')
print(grid.head())

,grid_id,geometry,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,"POLYGON ((423886.661 1340202.008, 423996.758 1...",1,12.122137,12.121729,12.122545,8.301005,8.300491,8.301519
1,1,"POLYGON ((425860.71 1334694.134, 425970.815 13...",2,12.072376,12.071968,12.072784,8.319272,8.318758,8.319786
2,2,"POLYGON ((427052.411 1338931.071, 427162.509 1...",3,12.110716,12.110308,12.111124,8.330126,8.329612,8.330640
3,3,"POLYGON ((427046.616 1338660.447, 427156.715 1...",4,12.108269,12.107861,12.108676,8.330079,8.329565,8.330593
4,4,"POLYGON ((427296.472 1329729.352, 427406.583 1...",5,12.027513,12.027105,12.027921,8.332575,8.332061,8.333088
...,...,...,...,...,...,...,...,...,...
167255,167255,"POLYGON ((475971.469 1330011.472, 476081.573 1...",167256,12.030775,12.030368,12.031183,8.779742,8.779228,8.780256
167256,167256,"POLYGON ((475969.606 1329921.277, 476079.71 13...",167257,12.029960,12.029552,12.030368,8.779726,8.779212,8.780240
167257,167257,"POLYGON ((475967.743 1329831.082, 476077.848 1...",167258,12.029144,12.028736,12.029552,8.779709,8.779195,8.780223
167258,167258,"POLYGON ((475965.88 1329740.888, 476075.985 13...",167259,12.028328,12.027921,12.028736,8.779693,8.779179,8.780207


## Building Footprint Data
Building footprint data is used to estimate population distribution within each 1km cell. We recommend using open-source building footprint data from the [Overture Map Foundation](https://overturemaps.org/). Building centroids are spatially joined to a 100 m resolution grid, and the number of buildings within each 100 m cell (bcount) is subsequently calculated.

In [ ]:
from pathlib import Path
import geopandas as gpd
import duckdb
import pyarrow.parquet as pq
import pyarrow as pa

In [ ]:
# Constants
RELEASE = "2026-01-21.0/"  # oventure data release date

data_inputs = Path("../Kano-scripts/Kano/data-inputs/").resolve()
data_inputs.mkdir(parents=True, exist_ok=True)

boundary_file = data_inputs / "grid-boundary-kano.gpkg"

out_bbox_parquet = data_inputs / "Building-footprint-Kano.parquet"
out_clip_parquet = data_inputs / "Building-footprint-Kano-clipped.parquet"
out_clip_geojson = data_inputs / "Building-footprint-Kano-clipped.geojson"

In [ ]:
# 1. Read the boundary file and get the study area's bounding box in EPSG:4326
gdf = gpd.read_file(boundary_file)
study_area = gdf.dissolve().reset_index(drop=True)

study_area_4326 = study_area.to_crs(4326)
minx, miny, maxx, maxy = study_area_4326.total_bounds
print("bbox:", minx, miny, maxx, maxy)

In [ ]:
# 2. Query the building footprints from Overture Maps using DuckDB and save the results as a Parquet file
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

src = f"s3://overturemaps-us-west-2/release/2026-01-21.0/theme=buildings/type=building/*"

query = f"""
SELECT id, geometry, bbox
FROM read_parquet('{src}', filename=true, hive_partitioning=1)
WHERE
  bbox.xmin <= {maxx} AND bbox.xmax >= {minx}
  AND bbox.ymin <= {maxy} AND bbox.ymax >= {miny}
"""

tbl = con.execute(query).fetch_arrow_table()
df = tbl.to_pandas()

# Convert the WKB geometry to GeoDataFrame and set the CRS to EPSG:4326
buildings_4326 = gpd.GeoDataFrame(
    df.drop(columns=["bbox"]),
    geometry=gpd.GeoSeries.from_wkb(df["geometry"]),
    crs=4326
)
print("downloaded rows:", len(buildings_4326))

# Save the building footprints as a Parquet file
buildings_4326.to_parquet(out_bbox_parquet)
print("Saved bbox parquet:", out_bbox_parquet)

In [ ]:
# Clip the building footprints to the study area and save the results as both Parquet and GeoJSON files
buildings = buildings_4326.to_crs(study_area.crs)
clipped = gpd.clip(buildings, study_area)

clipped.to_parquet(out_clip_parquet)
clipped.to_crs(4326).to_file(out_clip_geojson, driver="GeoJSON")

print("Saved clipped parquet:", out_clip_parquet)
print("Saved clipped geojson:", out_clip_geojson)

In [ ]:
# Count buildings per grid cell
# Loading Google building footprints
building_file = data_inputs / 'Kano_GOBv3.gpkg'
buildings = gpd.read_file(building_file)
buildings = buildings.to_crs(epsg)
buildings['centroid'] = buildings['geometry'].centroid

# Joining buildings to grid
grid_buildings = grid.sjoin(buildings.set_geometry('centroid').drop(columns='geometry'), how='inner', predicate='intersects')
grid_buildings = grid_buildings.groupby('grid_id')

# Counting buildings per grid
building_counts = grid_buildings.size().rename('bcount')

# Adding building count to grid cells
grid = grid.merge(building_counts, on='grid_id', how='left')

# Assign building count 0 to cells with no buildings (NaN)
grid['bcount'] = grid['bcount'].fillna(0)
grid.head()

The population of each 1km grid is distributed to underlying 100m cells proportionally based on building density. Each 100m grid is assigned a weight equal to its share of the total building count within the 1km grid.

In [ ]:
# Adding population data at 1km grid to finer grid

data_path = Path(data_inputs)

# Loading coarse pop data
pop_file = data_path / 'kano_nga_f_15_49_2015_1km.tif'
pop_raster, transform, crs = read_tif(pop_file)

# Converting the raster grid to vector data
pop_grid = raster2vector(pop_raster, transform, crs) # rember to save this as a geopackage for future use
pop_grid = pop_grid.to_crs(epsg)
# pop_grid.to_file(data_inputs + 'kano_pop_grid_1km.gpkg', driver='GPKG')

pop_grid['pop_grid_id'] = range(len(pop_grid))

grid = gpd.sjoin(grid.set_geometry('centroid'), pop_grid[['pop_grid_pop', 'geometry']], how='left', predicate='within')

# Assign coarse population data to finer grid based on the centroid locations of the finer grid cells
grid['centroid'] = grid['geometry'].centroid
grid = gpd.sjoin(grid.set_geometry('centroid'), pop_grid, how='left', predicate='within')
print(grid.columns)
grid = grid[['grid_id', 'bcount', 'pop_grid_id', 'geometry','rowid', 'latitude', 'lat_min', 'lat_max',
       'longitude', 'lon_min', 'lon_max']]
grid.head()

In [ ]:
# Calculate population weight (fraction of total population count that should be assigned to cell based on its building count)
grid_grouped_pop = grid.groupby('pop_grid_id')
building_count_pop = grid_grouped_pop['bcount'].sum().rename('pop_grid_bcount')
grid = grid.merge(building_count_pop, on='pop_grid_id', how='left')
grid['pop_weight'] = grid['bcount'] / grid['pop_grid_bcount']

# Compute disaggregated population count based on weight and building count at coarser cell level
grid = grid.merge(pop_grid, on='pop_grid_id', how='left')
grid['pop'] = grid['pop_grid_pop'] * grid['pop_weight']
grid.head()

In [ ]:
# Saving to file
grid = grid.drop(columns=["geometry_y"])
grid = grid.set_geometry("geometry_x")
grid = grid.to_crs(4326)
grid.to_file(data_temp + 'pop-grid-kano-centroids.gpkg', driver='GPKG')

## Next Steps

Data is now prepared for E2SFCA analysis. Proceed to:

**→ Notebook 02: E2SFCA Analysis** (`E2SFCA_Analysis.ipynb`)

This notebook will:
1. Calculate distance matrices
2. Run E2SFCA analysis for each facility category
3. Calculate accessibility scores
4. Classify deprivation levels